# Setup and Load Processed Data

In [2]:
import os
import sys
import json
import numpy as np
import random
from collections import defaultdict
from sklearn.model_selection import train_test_split
import tensorflow as tf

sys.path.append(os.path.abspath('..'))
from src.lstm_proximity import build_lstm_proximity_model
from src.bm25_ranker import BM25Engine
from src.preprocessing import IRPreprocessor
from src.metrics import calculate_average_precision, calculate_map

RAW_DIR = '../data/raw/'
PROCESSED_DIR = '../data/processed/'

doc_sequences = np.load(os.path.join(PROCESSED_DIR, 'doc_sequences.npy'))
query_sequences = np.load(os.path.join(PROCESSED_DIR, 'query_sequences.npy'))
embedding_matrix = np.load(os.path.join(PROCESSED_DIR, 'embedding_matrix.npy'))

with open(os.path.join(RAW_DIR, 'cran_docs.json'), 'r') as f:
    docs = json.load(f)
with open(os.path.join(RAW_DIR, 'cran_queries.json'), 'r') as f:
    queries = json.load(f)
with open(os.path.join(RAW_DIR, 'cran_qrels.json'), 'r') as f:
    qrels = json.load(f)

print(f"Loaded {len(doc_sequences)} vectorized documents and {len(query_sequences)} vectorized queries.")

Loaded 1400 vectorized documents and 226 vectorized queries.


# Split Dataset

In [3]:
ground_truth = defaultdict(set)
for qrel in qrels:
    q_id = str(int(qrel.get('query_number', qrel.get('query number', qrel.get('id', 0)))))
    d_id = str(int(qrel['id']))
    ground_truth[q_id].add(d_id)

# Split queries into Train (80%) and Test (20%)
all_query_ids = list(ground_truth.keys())
train_q_ids, test_q_ids = train_test_split(all_query_ids, test_size=0.2, random_state=42)

print(f"Training on {len(train_q_ids)} queries. Testing MAP on {len(test_q_ids)} queries.")

X_train_queries = []
X_train_docs = []
y_train = []

doc_ids = [str(int(doc.get('id', i+1))) for i, doc in enumerate(docs)]
doc_id_to_index = {d_id: i for i, d_id in enumerate(doc_ids)}

for q_idx, query in enumerate(queries):
    q_id = str(int(query.get('query_number', query.get('query number', query.get('id', q_idx+1)))))
    
    if q_id not in train_q_ids:
        continue # Save this query for the test set
        
    relevant_docs = list(ground_truth[q_id])
    
    for d_id in relevant_docs:
        if d_id in doc_id_to_index:
            # Positive sample
            X_train_queries.append(query_sequences[q_idx])
            X_train_docs.append(doc_sequences[doc_id_to_index[d_id]])
            y_train.append(1)
            
            # Generate a random negative sample for balance
            random_d_id = random.choice(doc_ids)
            while random_d_id in relevant_docs:
                random_d_id = random.choice(doc_ids)
                
            X_train_queries.append(query_sequences[q_idx])
            X_train_docs.append(doc_sequences[doc_id_to_index[random_d_id]])
            y_train.append(0)

X_train_queries = np.array(X_train_queries)
X_train_docs = np.array(X_train_docs)
y_train = np.array(y_train)

print(f"Generated {len(y_train)} training pairs (50% positive, 50% negative).")

Training on 739 queries. Testing MAP on 185 queries.
Generated 214 training pairs (50% positive, 50% negative).


# Training

In [4]:
vocab_size = embedding_matrix.shape[0]
embedding_dim = embedding_matrix.shape[1]
max_seq_length = doc_sequences.shape[1]

# Build the LSTM architecture
model = build_lstm_proximity_model(vocab_size=vocab_size, embedding_dim=embedding_dim, max_len=max_seq_length)

# Inject pre-trained embeddings and freeze them
model.layers[2].set_weights([embedding_matrix])
model.layers[2].trainable = False 
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Training
print("Starting training...")
history = model.fit(
    [X_train_queries, X_train_docs], 
    y_train, 
    epochs=10, 
    batch_size=32, 
    validation_split=0.2
)

Starting training...
Epoch 1/10
6/6 [==============================] - 16s 832ms/step - loss: 0.7743 - accuracy: 0.5029 - val_loss: 0.7694 - val_accuracy: 0.4884
Epoch 2/10
6/6 [==============================] - 3s 605ms/step - loss: 0.7228 - accuracy: 0.5088 - val_loss: 0.7507 - val_accuracy: 0.4884
Epoch 3/10
6/6 [==============================] - 2s 354ms/step - loss: 0.6982 - accuracy: 0.5029 - val_loss: 0.7478 - val_accuracy: 0.4884
Epoch 4/10
6/6 [==============================] - 3s 459ms/step - loss: 0.6850 - accuracy: 0.5673 - val_loss: 0.7402 - val_accuracy: 0.4884
Epoch 5/10
6/6 [==============================] - 2s 343ms/step - loss: 0.6769 - accuracy: 0.5965 - val_loss: 0.7325 - val_accuracy: 0.5116
Epoch 6/10
6/6 [==============================] - 2s 371ms/step - loss: 0.6624 - accuracy: 0.5848 - val_loss: 0.7338 - val_accuracy: 0.5349
Epoch 7/10
6/6 [==============================] - 2s 315ms/step - loss: 0.6488 - accuracy: 0.6023 - val_loss: 0.7312 - val_accuracy: 0.511

# Hybrid Scoring Fusion & MAP Evaluation

In [5]:
preprocessor = IRPreprocessor()
tokenized_corpus = [preprocessor.tokenize_for_bm25(str(doc.get('body', ''))) for doc in docs]
bm25_engine = BM25Engine(tokenized_corpus)

alpha = 0.7 # 70% BM25 base, 30% LSTM dependency adjustment
test_ap_scores = []
baseline_ap_scores = []

print("Evaluating on Test Set...")

for q_idx, query in enumerate(queries):
    q_id = str(int(query.get('query_number', query.get('query number', query.get('id', q_idx+1)))))
    
    if q_id not in test_q_ids:
        continue
        
    q_text = str(query.get('query', ''))
    tokenized_query = preprocessor.tokenize_for_bm25(q_text)
    
    # Base Probabilistic Score
    bm25_scores = bm25_engine.get_scores(tokenized_query)
    max_bm25 = np.max(bm25_scores) if np.max(bm25_scores) > 0 else 1
    bm25_scores_norm = bm25_scores / max_bm25
    
    # LSTM Proximity Score
    q_seq = np.tile(query_sequences[q_idx], (len(docs), 1))
    lstm_scores = model.predict([q_seq, doc_sequences], verbose=0).flatten()
    
    # Hybrid Fusion
    hybrid_scores = (alpha * bm25_scores_norm) + ((1 - alpha) * lstm_scores)
    
    # Evaluate Baseline (BM25 only)
    base_ranked_indices = np.argsort(bm25_scores)[::-1]
    base_retrieved_ids = [doc_ids[i] for i in base_ranked_indices]
    baseline_ap_scores.append(calculate_average_precision(base_retrieved_ids, ground_truth[q_id]))
    
    # Evaluate Hybrid
    hybrid_ranked_indices = np.argsort(hybrid_scores)[::-1]
    hybrid_retrieved_ids = [doc_ids[i] for i in hybrid_ranked_indices]
    test_ap_scores.append(calculate_average_precision(hybrid_retrieved_ids, ground_truth[q_id]))

print(f"Isolated Test Set MAP (Baseline BM25): {calculate_map(baseline_ap_scores):.4f}")
print(f"Isolated Test Set MAP (LSTM-Augmented): {calculate_map(test_ap_scores):.4f}")

Evaluating on Test Set...
Isolated Test Set MAP (Baseline BM25): 0.0024
Isolated Test Set MAP (LSTM-Augmented): 0.0037
